# Demo Visual: Overlay de Ruido sobre Detecciones

Este notebook demuestra cómo podemos tomar una imagen, detectar objetos con YOLOv8 OBB, y luego "intervenir" la imagen aplicando un parche de ruido sobre las detecciones. Esto simula el primer paso del ataque adversarial físico antes de iniciar la optimización matemática.

In [ ]:
# 1. Instalar dependencias necesarias (Descomentar si estás en Colab)
# !pip install ultralytics opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import urllib.request

In [ ]:
# 2. Descargar una imagen aérea de prueba y cargar el modelo
image_url = "https://ultralytics.com/images/dota8.jpg" # Imagen de prueba compatible con DOTA
image_path = "sample_dota.jpg"
urllib.request.urlretrieve(image_url, image_path)

# Cargar el modelo pre-entrenado para detección orientada
model = YOLO("yolov8n-obb.pt")

# Leer imagen original
img_bgr = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(img_rgb)
plt.title("Imagen Original")
plt.axis('off')
plt.show()

In [ ]:
# 3. Correr Baseline: Detección Original
results_original = model(img_rgb)

# Visualizar detecciones originales
res_img_original = results_original[0].plot()

plt.figure(figsize=(10, 10))
plt.imshow(res_img_original)
plt.title("Detecciones Baseline (YOLOv8 OBB)")
plt.axis('off')
plt.show()

In [ ]:
# 4. Generar Overlay (Ruido Aleatorio) sobre los objetos detectados
img_attacked = img_rgb.copy()

if results_original[0].obb is not None:
    # Obtener coordenadas de los polígonos (4 puntos por objeto)
    boxes = results_original[0].obb.xyxyxyxy.cpu().numpy()
    
    for box in boxes:
        pts = np.array(box, np.int32)
        pts = pts.reshape((-1, 1, 2))
        
        # Crear un parche de ruido del tamaño de la imagen
        noise = np.random.randint(0, 256, img_attacked.shape, dtype=np.uint8)
        
        # Crear una máscara vacía
        mask = np.zeros_like(img_attacked)
        
        # Rellenar el polígono en la máscara con blanco
        cv2.fillPoly(mask, [pts], (255, 255, 255))
        
        # Aplicar el ruido solo en el área de la máscara
        img_attacked = np.where(mask == 255, noise, img_attacked)

plt.figure(figsize=(10, 10))
plt.imshow(img_attacked)
plt.title("Imagen con 'Camuflaje' (Ruido Aleatorio) Aplicado")
plt.axis('off')
plt.show()

In [ ]:
# 5. Volver a evaluar con YOLOv8 OBB
results_attacked = model(img_attacked)

res_img_attacked = results_attacked[0].plot()

plt.figure(figsize=(10, 10))
plt.imshow(res_img_attacked)
plt.title("Detecciones después del Overlay Aleatorio")
plt.axis('off')
plt.show()

# Comparación rápida
num_orig = len(results_original[0]) if results_original[0].obb is not None else 0
num_atk = len(results_attacked[0]) if results_attacked[0].obb is not None else 0
print(f"Objetos detectados originalmente: {num_orig}")
print(f"Objetos detectados tras aplicar ruido: {num_atk}")
print("Nota: El ruido aleatorio usualmente empeora la detección, pero no siempre es un ataque perfecto. \nEl objetivo del algoritmo será optimizar este parche.")